# Figures for the two-dimensional models

Reproduces the figures of the paper on the two-dimensional models from the processed data in
`reference/` and `results/`:

| Figure | Content | Data produced by |
|---|---|---|
| Fig. 2 (main text) | Bi-stable double well: PES, $p(x|\mathrm{TP})$, path overlap, channel autocorrelation | `bistable_double_well_analysis.ipynb`, `bistable_double_well_path_overlap.ipynb` |
| Fig. S1 | Standard double well: PES, $p(x|\mathrm{TP})$, convergence to the reference | `standard_double_well_analysis.ipynb` |
| Fig. S2 | Bi-stable double well: convergence to the reference | `scripts/` (cluster runs, see `README.md`) |
| Fig. S6 | Path length distributions | `standard_double_well_analysis.ipynb`, `bistable_double_well_analysis.ipynb` |

Figures are saved in `figures/`.

In [ ]:
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.lines as mlines
from matplotlib.colors import ListedColormap
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from mpl_toolkits.axes_grid1 import make_axes_locatable

from aaa_tps.systems.two_dimensional_double_wells import StandardDoubleWell, BistableDoubleWell
from aaa_tps.states.states import circular_state

mpl.rcParams.update({'font.size': 9})
cm_to_inch = 0.393701
dpi = 400

os.makedirs("./figures", exist_ok=True)

## Methods, colors and data files

In [ ]:
methods = {
    "2w_Gaus": dict(dir="twoway_shooting/{model}/Gaussian", run="2w", label="Two-Way (Gaussian)", color="C0"),
    "1w_Gaus": dict(dir="oneway_shooting/{model}/Gaussian", run="1w", label="One-Way (Gaussian)", color="C1"),
    "ARA_TPS_Unif": dict(dir="ARA_TPS/{model}/Uniform", run="ARA-TPS", label="ARA-TPS (Uniform)", color="C4"),
    "ARA_TPS_Gaus": dict(dir="ARA_TPS/{model}/Gaussian", run="ARA-TPS", label="ARA-TPS (Gaussian)", color="C5"),
    "AAA_TPS_Unif": dict(dir="AAA_TPS/{model}/Uniform", run="AAA-TPS", label="AAA-TPS (Uniform)", color="C2"),
    "AAA_TPS_Gaus": dict(dir="AAA_TPS/{model}/Gaussian", run="AAA-TPS", label="AAA-TPS (Gaussian)", color="C3"),
}

# Order of the curves in the convergence plots (sets the color cycle as in the paper)
convergence_order = ["2w_Gaus", "1w_Gaus", "AAA_TPS_Unif", "AAA_TPS_Gaus", "ARA_TPS_Unif", "ARA_TPS_Gaus"]


def result_file(key, model, name):
    m = methods[key]
    return os.path.join("./results", m["dir"].format(model=model), f"{name}_{m['run']}.txt")


def legend_handles(keys):
    return [mlines.Line2D([], [], color=methods[k]["color"], label=methods[k]["label"]) for k in keys]

## Potential energy surfaces and stable states

In [ ]:
def pes_on_grid(system, state_A, state_B, x_values, y_values):
    x_grid, y_grid = np.meshgrid(x_values, y_values)
    U = np.zeros(x_grid.shape)
    in_A = np.zeros(x_grid.shape)
    in_B = np.zeros(x_grid.shape)
    for i in range(x_grid.shape[0]):
        for j in range(x_grid.shape[1]):
            x = np.array([x_grid[i, j], y_grid[i, j]])
            U[i, j] = system.energy_function(x)
            in_A[i, j] = state_A.is_in_state(x)
            in_B[i, j] = state_B.is_in_state(x)
    return x_grid, y_grid, U - U.min(), in_A, in_B


stdw = StandardDoubleWell(barrier=10, asymmetry=0)
stdw_state_A = circular_state(center=(-1,-1), axes=(1,2), angle=-0.25, radius=0.05)
stdw_state_B = circular_state(center=(1,1), axes=(1,2), angle=-0.25, radius=0.05)
stdw_x_grid, stdw_y_grid, stdw_U, stdw_in_state_A, stdw_in_state_B = pes_on_grid(
    stdw, stdw_state_A, stdw_state_B, np.linspace(-1.6, 1.6, 500), np.linspace(-2.6, 2.6, 500))

bsdw = BistableDoubleWell(A=15, B=0.25)
bsdw_state_A = circular_state(center=(-2.,0), axes=(1,1), angle=0, radius=0.15)
bsdw_state_B = circular_state(center=(2.,0), axes=(1,1), angle=0, radius=0.15)
bsdw_x_grid, bsdw_y_grid, bsdw_U, bsdw_in_state_A, bsdw_in_state_B = pes_on_grid(
    bsdw, bsdw_state_A, bsdw_state_B, np.linspace(-3., 3., 500), np.linspace(-3., 3., 500))

In [ ]:
white_cmap = ListedColormap(["white"])


def draw_states(ax, x_grid, y_grid, in_A, in_B, pos_A, pos_B, **text_kw):
    """Draw the stable states as white regions with their labels."""
    for in_state, pos, name in ((in_A, pos_A, "A"), (in_B, pos_B, "B")):
        mesh = ax.pcolormesh(x_grid, y_grid, in_state, cmap=white_cmap, alpha=1, shading='nearest')
        mesh.set_array(np.where(in_state, 1, np.nan).ravel())
        mesh.set_clim(0, 1)
        ax.text(*pos, name, fontweight='normal', transform=ax.transAxes, c="k", **text_kw)


def draw_pes(fig, ax, x_grid, y_grid, U, levels, ticks):
    im = ax.contourf(x_grid, y_grid, U, levels=levels, cmap="RdBu_r")
    cax = make_axes_locatable(ax).append_axes("top", size="5%", pad=0.04)
    cbar = fig.colorbar(im, cax=cax, orientation='horizontal', ticks=ticks)
    cbar.ax.xaxis.set_label_position('top')
    cbar.ax.xaxis.tick_top()
    cbar.set_ticklabels([r"$0$"] + [rf"${t}\,k_BT$" for t in ticks[1:]])
    cbar.ax.tick_params(labelsize=8)

## Fig. 2 (main text): bi-stable double well

(A) Potential energy surface and stable states. (B) Reference $p(x|\mathrm{TP})$.
(C) Average fractional overlap between successive trajectories; dotted lines: average force evaluations
for a complete renewal of the trajectory. (D) Autocorrelation of the transition channel
(reweighted for AAA-TPS).

In [ ]:
bsdw_Hist = np.loadtxt("./reference/twoway_shooting/bsdw/Uniform/Hist_ref_avg_for_plotting.txt")

fig = plt.figure(figsize=(18*cm_to_inch, 7.*cm_to_inch), dpi=dpi)
gs = gridspec.GridSpec(2, 3, figure=fig, height_ratios=[.05, 1], width_ratios=[.75, 1, 1], hspace=0.6, wspace=0.5)

ax_legend = fig.add_subplot(gs[0, 1:])
ax_legend.axis("off")
ax_legend.add_artist(ax_legend.legend(handles=legend_handles(methods), loc='center', bbox_to_anchor=(0.5, 1.0),
                                      ncol=2, frameon=False, fontsize=9, borderaxespad=-8.0))

subgrid = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=gs[:, 0], wspace=0.1, hspace=0.2)
axA = fig.add_subplot(subgrid[0, 0])
axB = fig.add_subplot(subgrid[1, 0])
axC = fig.add_subplot(gs[1, 1])
axD = fig.add_subplot(gs[1, 2])
plt.setp(axA.get_xticklabels(), visible=False)

# (A) PES
axA.text(-0.4, 1.25, "A", fontsize=15, transform=axA.transAxes, c="k")
draw_pes(fig, axA, bsdw_x_grid, bsdw_y_grid, bsdw_U, np.linspace(0, 11, 21), [0, 5, 10])
draw_states(axA, bsdw_x_grid, bsdw_y_grid, bsdw_in_state_A, bsdw_in_state_B, (0.17, 0.47), (0.78, 0.47), fontsize=6)
axA.set_xlim(-3.3, 3.3)
axA.set_ylim(-2.5, 2.5)
aspect_bsdw = axA.get_aspect()
axA.set_ylabel(r"$x^1$")
axA.set_yticks([-2, 0, 2])

# (B) Reference p(x|TP)
axB.text(-0.4, 1.05, "B", fontsize=15, transform=axB.transAxes, c="k")
axB.imshow(bsdw_Hist.T, origin='lower', extent=[-3.3, 3.3, -3.3, 3.3], cmap='hot', interpolation="bicubic")
draw_states(axB, bsdw_x_grid, bsdw_y_grid, bsdw_in_state_A, bsdw_in_state_B, (0.17, 0.47), (0.78, 0.47), fontsize=6)
axB.contour(bsdw_x_grid, bsdw_y_grid, bsdw_U, levels=np.linspace(0, 11, 6), colors='white', linewidths=0.5, linestyles='dashed')
axB.set_ylim(-2.5, 2.5)
axB.set_aspect(aspect_bsdw)
axB.set_yticks([-2, 0, 2])
axB.set_xlabel(r'$x^0$')
axB.set_ylabel(r'$x^1$')

# (C) Path overlap
axC.text(-0.2, 1.1, "C", fontsize=15, transform=axC.transAxes, c="k")
axC.axhline(y=0, color="k", ls="--", linewidth=1, alpha=0.5)
for zorder, (key, m) in zip(range(10, 4, -1), methods.items()):
    f_eval, overlap = np.loadtxt(result_file(key, "bsdw", "OPL_vs_feval"), unpack=True)
    renewal = np.loadtxt(result_file(key, "bsdw", "OPL_renewal"))[0]
    axC.plot(f_eval, overlap, color=m["color"], zorder=zorder)
    axC.axvline(renewal, color=m["color"], ls=":")
axC.set_xscale("log")
axC.set_ylim(-0.1, 1)
axC.set_xlim(1e2, 1e5)
axC.set_xlabel(r'Force Evaluations $n_{\text{F}}$')
axC.set_ylabel('Fractional Path Overlap')

# (D) Channel autocorrelation (last column: reweighted for AAA-TPS)
axD.text(-0.2, 1.1, "D", fontsize=15, transform=axD.transAxes, c="k")
axD.axhline(y=0, color="k", ls="--", linewidth=1, alpha=0.5)
for key, m in methods.items():
    data = np.loadtxt(result_file(key, "bsdw", "ch_correlation"))
    axD.plot(data[:, 0], data[:, -1], color=m["color"])
axD.set_xlabel(r"Force Evaluations $n_\text{F}$")
axD.set_ylabel(r"$C_{\text{ch}}(n_{\text{F}})$")
axD.set_xlim(1e3, 1e6)
axD.set_ylim(-0.1, 1)
axD.set_xscale("log")

for ax in [axA, axB, axC, axD]:
    ax.tick_params(labelsize=8)

plt.tight_layout()
plt.savefig("./figures/Fig2_bistable_double_well.png", dpi=dpi, bbox_inches="tight")
plt.show()

## Fig. S1: standard double well

(A) Potential energy surface and stable states. (B) Reference $p(x|\mathrm{TP})$. (C) Convergence of
$p(x|\mathrm{TP})$ to the reference as a function of the force evaluations (main) and of the TPS trials
(inset). For AAA-TPS the reweighted (resampled) ensemble is shown, i.e. the last column of the files.

In [ ]:
stdw_Hist = np.loadtxt("./reference/twoway_shooting/stdw/Uniform/Hist_ref_avg_for_plotting.txt")
traj_step = 1000

# As in the paper, the distances are divided by the sum of the reference histogram on the plotting grid
norm_conv = stdw_Hist.sum()

fig = plt.figure(figsize=(18*cm_to_inch, 7.*cm_to_inch), dpi=dpi)
gs = gridspec.GridSpec(2, 2, figure=fig, height_ratios=[.05, 1], width_ratios=[.5, 1], hspace=0.6, wspace=0.5)

ax_legend = fig.add_subplot(gs[0, 1:])
ax_legend.axis("off")
ax_legend.add_artist(ax_legend.legend(handles=legend_handles(methods), loc='center', bbox_to_anchor=(0.4, 1.0),
                                      ncol=2, frameon=False, fontsize=9, borderaxespad=-8.0))

subgrid = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=gs[:, 0], wspace=0.1, hspace=0.2)
axA = fig.add_subplot(subgrid[0, 0])
axB = fig.add_subplot(subgrid[1, 0])
axC = fig.add_subplot(gs[1, 1])
plt.setp(axA.get_xticklabels(), visible=False)

# (A) PES
axA.text(-0.2, 1., "A", fontsize=15, transform=axA.transAxes, c="k")
draw_pes(fig, axA, stdw_x_grid, stdw_y_grid, stdw_U, np.linspace(0, 22, 22), [0, 10, 20])
draw_states(axA, stdw_x_grid, stdw_y_grid, stdw_in_state_A, stdw_in_state_B, (0.15, 0.26), (0.775, 0.65))
axA.set_xlim(-1.6, 1.6)
axA.set_ylim(-2.6, 2.6)
aspect_stdw = axA.get_aspect()
axA.set_ylabel(r"$x^1$")

# (B) Reference p(x|TP)
axB.text(-0.2, 1., "B", fontsize=15, transform=axB.transAxes, c="k")
axB.imshow(stdw_Hist.T, origin='lower', extent=[-1.6, 1.6, -2.6, 2.6], cmap='hot', interpolation='bicubic')
axB.set_aspect(aspect_stdw)
draw_states(axB, stdw_x_grid, stdw_y_grid, stdw_in_state_A, stdw_in_state_B, (0.15, 0.26), (0.775, 0.65))
axB.contour(stdw_x_grid, stdw_y_grid, stdw_U, levels=np.linspace(0, 22, 6), colors='white', linewidths=0.5, linestyles='dashed')
axB.set_xlabel(r'$x^0$')
axB.set_ylabel(r'$x^1$')
axB.set_yticks([-2, 0, 2])

# (C) Convergence
axC.text(-0.09, 1.05, "C", fontsize=15, transform=axC.transAxes, c="k")
axC_inset = inset_axes(axC, width="60%", height="60%", loc='upper right', bbox_to_anchor=(0.0, 0.0, 1, 1), bbox_transform=axC.transAxes)
for key in convergence_order:
    data = np.loadtxt(result_file(key, "stdw", "P_x_TP_conv"))
    trials = np.arange(len(data))*traj_step
    axC.plot(data[:, 0]/1e9, data[:, -1]/norm_conv, color=methods[key]["color"])
    axC_inset.plot(trials/1e6, data[:, -1]/norm_conv, color=methods[key]["color"], lw=.75)
axC.set_yscale("log")
axC.set_xlabel(r"Force Evaluations $n_\text{F}$ ($\times 10^9$)")
axC.set_ylabel(r"$\sum_{b\in\text{bins}} |p^{(b)}(x|TP) - p^{(b)}_{\text{ref}}(x|TP)|$")
axC_inset.set_yscale("log")
axC_inset.set_xticks([0, 5])
axC_inset.set_xlabel(r"Trials $\tau$ ($\times 10^6$)")
axC_inset.xaxis.set_label_coords(0.5, -0.066)
axC_inset.tick_params(labelsize=8)

for ax in [axA, axB, axC]:
    ax.tick_params(labelsize=8)

plt.tight_layout()
plt.savefig("./figures/FigS1_standard_double_well.png", dpi=dpi, bbox_inches="tight")
plt.show()

## Fig. S2: convergence for the bi-stable double well

Convergence of $p(x|\mathrm{TP})$ (2400 replicas of $10^5$ trials per method) to the reference
(24 675 replicas of two-way shooting with uniform selection probability, symmetrized along $x^1$).
Left: sampled ensembles; right: sampled ensembles symmetrized along $x^1$. Insets: same data as a
function of the TPS trials.

In [ ]:
fig = plt.figure(figsize=(18*cm_to_inch, 7.5*cm_to_inch), dpi=dpi)
gs = gridspec.GridSpec(2, 2, figure=fig, height_ratios=[.05, 1], width_ratios=[1, 1], hspace=0.45, wspace=0.4)

ax_legend = fig.add_subplot(gs[0, :])
ax_legend.axis("off")
ax_legend.add_artist(ax_legend.legend(handles=legend_handles(methods), loc='center', bbox_to_anchor=(0.485, -1.3),
                                      ncol=3, frameon=False, fontsize=9, borderaxespad=0))

axA = fig.add_subplot(gs[1, 0])
axB = fig.add_subplot(gs[1, 1])

for ax, name, ylabel in ((axA, "convergence_l1", r"$\sum_{b\in\text{bins}} |p^{(b)}(x|TP) - p^{(b)}_{\text{ref}}(x|TP)|$"),
                         (axB, "convergence_l1_sym", r"$\sum_{b\in\text{bins}} |\tilde{p}^{(b)}(x|TP) - p^{(b)}_{\text{ref}}(x|TP)|$")):
    ax_inset = inset_axes(ax, width="60%", height="60%", loc='upper right', bbox_to_anchor=(0.0, 0.0, 0.98, 1), bbox_transform=ax.transAxes)
    for key in convergence_order:
        count, conv, feval = np.loadtxt(result_file(key, "bsdw", name), unpack=True)
        ax.plot(feval/1e11, conv, color=methods[key]["color"])
        ax_inset.plot(count*1e5/1e8, conv, color=methods[key]["color"], lw=.75)
    ax.set_yscale("log")
    ax.set_xlabel(r"Force Evaluations $n_\text{F}$ ($\times 10^{11}$)")
    ax.set_ylabel(ylabel)
    ax_inset.set_yscale("log")
    ax_inset.set_xticks([0, 2.5])
    ax_inset.set_xlabel(r"Trials $\tau$ ($\times 10^8$)")
    ax_inset.xaxis.set_label_coords(0.5, -0.066)
    ax_inset.tick_params(labelsize=8)
    ax.tick_params(labelsize=8)

plt.tight_layout()
plt.savefig("./figures/FigS2_bistable_double_well_convergence.png", dpi=dpi, bbox_inches="tight")
plt.show()

## Fig. S6: path length distributions

Reference, methods sampling the transition path ensemble, and AAA-TPS before (Unrew.) and after
reweighting.

In [ ]:
fig = plt.figure(figsize=(18*cm_to_inch, 8*cm_to_inch), dpi=dpi)
gs = gridspec.GridSpec(2, 2, figure=fig, height_ratios=[.05, 1], width_ratios=[1, 1], hspace=0.8, wspace=0.4)

handles = [mlines.Line2D([], [], color='k', label='Reference')] + legend_handles(["2w_Gaus", "1w_Gaus", "ARA_TPS_Unif", "ARA_TPS_Gaus"])
handles += [mlines.Line2D([], [], color='C2', ls=":", label='Unrew. (Uniform)'), mlines.Line2D([], [], color='C2', label='AAA-TPS (Uniform)'),
            mlines.Line2D([], [], color='C3', ls=":", label='Unrew. (Gaussian)'), mlines.Line2D([], [], color='C3', label='AAA-TPS (Gaussian)')]

ax_legend = fig.add_subplot(gs[0, :])
ax_legend.axis("off")
ax_legend.add_artist(ax_legend.legend(handles=handles, loc='center', bbox_to_anchor=(0.485, -1.3),
                                      ncol=3, frameon=False, fontsize=9, borderaxespad=0))

for col, (model, title) in enumerate((("stdw", "Standard Double Well"), ("bsdw", "Bi-Stable Double Well"))):
    ax = fig.add_subplot(gs[1, col])
    fname = f"./results/path_lengths/{model}_P_L.txt"
    names = open(fname).readline().lstrip("#").split()
    data = dict(zip(names, np.loadtxt(fname, unpack=True)))
    bin_centers = (data["bin_left"] + data["bin_right"])/2

    ax.plot(bin_centers, data["Reference"], color="k")
    for key in ["2w_Gaus", "1w_Gaus", "ARA_TPS_Unif", "ARA_TPS_Gaus"]:
        ax.plot(bin_centers, data[key], color=methods[key]["color"])
    for key in ["AAA_TPS_Unif", "AAA_TPS_Gaus"]:
        ax.plot(bin_centers, data[f"{key}_unrew"], color=methods[key]["color"], ls=":")
        ax.plot(bin_centers, data[key], color=methods[key]["color"])

    ax.set_xlabel(r"$L$")
    ax.set_ylabel(r"$P(L)$")
    ax.set_xlim(0, data["bin_right"][-1])
    ax.set_title(title)
    ax.tick_params(labelsize=8)

plt.savefig("./figures/FigS6_path_length_distributions.png", dpi=dpi, bbox_inches="tight")
plt.show()